목표

기존 방식: 키워드 포함 여부로 분류

문제:

표현이 다양하면 못 잡음

오타/은어/문맥을 못 이해함

기타가 너무 많음

LLM 방식: 

리뷰 내용을 읽고 의미 기준으로 분류

In [ ]:
# =========================================================
# 01. 실행 설정
# =========================================================

RUN_LLM = True          # LLM 실행 여부
RUN_SAMPLE = True       # 샘플만 돌릴지 여부

TEST_N = 30             # 샘플 개수
RANDOM_STATE = 42       # 랜덤 시드

MIN_REVIEW_LEN = 50     # LLM에 넣을 최소 리뷰 길이
SLEEP_SEC = 0.5         # API 요청 간 대기 시간

SAVE_PATH = "../../../../data/preprocessed/llm_review_results.csv"
CHECKPOINT_INTERVAL = 20  # 중간 저장 주기

print("설정 완료")

In [1]:
# =========================================================
# 02. 데이터 준비
# =========================================================

df_base = reviews_en.copy()

# 리뷰 길이 컬럼 생성
df_base["review_len"] = df_base["review"].astype(str).str.len()

# LLM 대상: 기타 + 길이 조건
df_llm_target = df_base[
    (df_base["issue_categories_v2"].apply(lambda x: x == ["기타"])) &
    (df_base["review_len"] >= MIN_REVIEW_LEN)
].copy()

print("전체 리뷰:", len(df_base))
print("LLM 대상 리뷰:", len(df_llm_target))

df_llm_target[["review", "issue_categories_v2", "review_len"]].head()

NameError: name 'reviews_en' is not defined

In [ ]:
# =========================================================
# 03. 샘플 데이터
# =========================================================

if RUN_SAMPLE:
    df_llm_target = df_llm_target.sample(
        TEST_N,
        random_state=RANDOM_STATE
    ).copy()

print("LLM 실행 대상:", len(df_llm_target))

In [ ]:
# =========================================================
# 04. LLM 설정
# =========================================================

import os
import json
import time
from tqdm import tqdm
import google.generativeai as genai

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-1.5-flash")

print("LLM 설정 완료")

In [ ]:
# =========================================================
# 05. 카테고리 정의
# =========================================================

ACTIONABLE_CATEGORIES = [
    "버그/크래시",
    "최적화/성능",
    "저장/진행 문제",
    "콘텐츠 부족",
    "볼륨/플레이타임",
    "엔딩/후반부",
    "업데이트/개발",
    "가격/가성비",
    "조작/UX",
    "설명/튜토리얼 부족",
    "난이도/밸런스",
    "멀티/서버",
    "아트/비주얼",
    "스토리/캐릭터",
    "음악/사운드",
    "게임플레이",
    "분위기/감성",
    "반복 플레이/중독성",
    "기타"
]

In [ ]:
# =========================================================
# 06. 프롬프트 정의
# =========================================================

SYSTEM_PROMPT = """
너는 Steam 인디 게임 리뷰를 분석하는 데이터 분석가다.

목표:
게임 제작자가 개선하거나 강화할 수 있는 요소를 분류한다.

규칙:
- 반드시 제공된 카테고리 중에서 선택
- 최대 3개까지 선택
- 단순 감상은 '기타'
- JSON만 출력

형식:
{
  "categories": ["카테고리1", "카테고리2"],
  "reason": "짧은 설명"
}
"""

In [ ]:
# =========================================================
# 07. LLM 분류 함수
# =========================================================

def classify_review_llm(review_text, sentiment):

    user_prompt = f"""
카테고리:
{ACTIONABLE_CATEGORIES}

감정:
{sentiment}

리뷰:
\"\"\"{review_text}\"\"\"

분류해줘.
"""

    try:
        response = model.generate_content(
            SYSTEM_PROMPT + "\n\n" + user_prompt
        )

        text = response.text.strip()

        result = json.loads(text)

    except Exception as e:
        result = {
            "categories": ["기타"],
            "reason": str(e)
        }

    return result

In [ ]:
# =========================================================
# 08. LLM 실행
# =========================================================

llm_results = []

if RUN_LLM:
    for i, (idx, row) in enumerate(
        tqdm(df_llm_target.iterrows(), total=len(df_llm_target))
    ):

        result = classify_review_llm(
            row["review"],
            row["sentiment"]
        )

        llm_results.append({
            "index": idx,
            "appid": row["appid"],
            "genre": row["primary_genre"],
            "sentiment": row["sentiment"],
            "review": row["review"],
            "llm_categories": result["categories"],
            "llm_reason": result["reason"]
        })

        # 체크포인트 저장
        if (i+1) % CHECKPOINT_INTERVAL == 0:
            pd.DataFrame(llm_results).to_csv(SAVE_PATH, index=False)

        time.sleep(SLEEP_SEC)

llm_df = pd.DataFrame(llm_results)

llm_df.to_csv(SAVE_PATH, index=False)

print("LLM 완료")

In [ ]:
# =========================================================
# 09. 결과 확인
# =========================================================

llm_df.head()

llm_df["llm_categories"].value_counts()

In [ ]:
# =========================================================
# 10. 카테고리 분석
# =========================================================

llm_df["llm_categories"] = llm_df["llm_categories"].apply(
    lambda x: x if isinstance(x, list) else ["기타"]
)

llm_exploded = llm_df.explode("llm_categories")

category_summary = (
    llm_exploded
    .groupby(["sentiment", "llm_categories"])
    .size()
    .reset_index(name="count")
    .sort_values(["sentiment", "count"], ascending=[True, False])
)

category_summary

In [ ]:
# =========================================================
# 10. 카테고리 분석
# =========================================================

llm_df["llm_categories"] = llm_df["llm_categories"].apply(
    lambda x: x if isinstance(x, list) else ["기타"]
)

llm_exploded = llm_df.explode("llm_categories")

category_summary = (
    llm_exploded
    .groupby(["sentiment", "llm_categories"])
    .size()
    .reset_index(name="count")
    .sort_values(["sentiment", "count"], ascending=[True, False])
)

category_summary

In [ ]:
# =========================================================
# 11. 장르별 분석
# =========================================================

genre_summary = (
    llm_exploded
    .groupby(["genre", "sentiment", "llm_categories"])
    .size()
    .reset_index(name="count")
    .sort_values(["genre", "sentiment", "count"], ascending=[True, True, False])
)

genre_summary.head(30)